In [0]:
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

df = spark.table("gold.aapl_stock_data").dropna().toPandas().sort_values("date")

features = ["open", "high", "low", "volume", "ma_7d", "ma_30d", "volatility_7d"]
target = "close"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)
print(f"MAE: {mae:.4f} | R2: {r2:.4f}")

# Build dashboard dataset
test_df = df.iloc[len(X_train):].copy()
test_df["pred_close"] = preds
test_df["abs_error"] = (test_df["close"] - test_df["pred_close"]).abs()
test_df["pct_error"] = test_df["abs_error"] / test_df["close"]

spark.createDataFrame(test_df).write.format("delta").mode("overwrite").saveAsTable("gold.aapl_predictions")